In [ ]:
ds = load_dataset("cxyzhang/pmc_llmExtraction_v10")
finalized_df = ds["train"].to_pandas()

In [ ]:
finalized_df["topic"] = finalized_df['pmcid'] + ":" + finalized_df["topic"]

In [ ]:
categories = ['Vitals_Hema', 'Neuro', 'CVS', 'RESP', 'EENT', 'GI', 'GU', 'DERM', 'MSK', 'ENDO', 'LYMPH', "Pregnancy", 'History', 'Lab_Image']

In [ ]:
def load_embeddings(embedding_dir="embeddings"):
    """
    Loads embeddings and metadata from NPZ files for multiple categories,
    ensuring that rows with NaN topics are dropped.
    """
    data_dict = {}

    for category in categories:
        file_path = os.path.join(embedding_dir, f"{category}_embeddings.npz")
        if os.path.exists(file_path):
            data = np.load(file_path, allow_pickle=True)

            # Convert to NumPy arrays
            embeddings = data["embeddings"]
            topics = data["topic"]
            ages = data["age"]
            sexes = data["sex"]

            # Count NaN topics
            num_nan_topics = np.isnan(topics).sum() if topics.dtype != object else (pd.Series(topics).isna()).sum()

            # Identify valid (non-NaN) topic indices
            valid_indices = ~pd.isna(topics)  # Mask where topic is NOT NaN

            # Filter embeddings and metadata
            embeddings = embeddings[valid_indices]
            topics = topics[valid_indices]
            ages = ages[valid_indices]
            sexes = sexes[valid_indices]

            # Store cleaned data
            data_dict[category] = {
                "embeddings": embeddings,
                "topic": topics,
                "age": ages,
                "sex": sexes
            }

            print(f"Loaded {category} embeddings with shape {embeddings.shape} (after dropping {num_nan_topics} NaNs in topic)")

        else:
            print(f"Warning: {file_path} not found. Skipping {category}.")

    return data_dict

# Load the embeddings after cleaning
embeddings_data = load_embeddings()


In [ ]:
embeddings_data

In [ ]:
def assign_test_cases_by_frequency(embedding_data, frequency_group='top', num_topics=100, num_test_cases=30, random_seed=40):
    """
    Selects a subset of test cases from specified frequency groups (top, middle, bottom),
    ensuring each topic appears only once in the final test set.
    This version ensures FAISS retrieval integrity and balanced category selection.

    Parameters:
    - embedding_data: Dictionary containing embeddings and metadata per category.
    - frequency_group: Frequency group to sample from ('top', 'middle', 'bottom').
    - num_topics: Number of topics to select from the frequency group.
    - num_test_cases: Number of test cases to return.
    - random_seed: Seed for reproducibility.

    Returns:
    - test_df: DataFrame with unique test cases.
    - query_indices: List of (category, index) tuples for FAISS exclusion.
    """

    np.random.seed(random_seed)

    # Step 1: Compute topic frequencies across all categories
    topic_counts = Counter()
    for data in embedding_data.values():
        topic_counts.update(data.get("topic", []))

    sorted_topics = sorted(topic_counts.items(), key=lambda x: x[1], reverse=True)
    total_topics = len(sorted_topics)

    # Step 2: Select topics based on frequency group
    if frequency_group == 'top':
        selected_topics = {topic for topic, _ in sorted_topics[:num_topics]}
    elif frequency_group == 'bottom':
        selected_topics = {topic for topic, _ in sorted_topics[-num_topics:]}
    elif frequency_group == 'middle':
        mid_start = (total_topics - num_topics) // 2
        selected_topics = {topic for topic, _ in sorted_topics[mid_start:mid_start + num_topics]}
    else:
        raise ValueError("frequency_group must be 'top', 'middle', or 'bottom'")

    # Step 3: Collect valid test cases across all categories (balanced sampling)
    test_cases = []
    query_indices = []  # For FAISS exclusion
    selected_topics_set = set()
    per_category_quota = num_test_cases // len(embedding_data)

    for category, data in embedding_data.items():
        # print(f"\nChecking '{category}' keys: {list(data.keys())}")

        embeddings = np.array(data.get("embeddings", []), dtype=object)
        topics = np.array(data.get("topic", []), dtype=object)
        ages = np.array(data.get("age", []), dtype=object)
        sexes = np.array(data.get("sex", []), dtype=object)

        
        valid_indices = [i for i, topic in enumerate(topics) if topic in selected_topics and topic not in selected_topics_set]
        np.random.shuffle(valid_indices)

        for idx in valid_indices[:per_category_quota]:
            if idx < len(embeddings) and isinstance(embeddings[idx], np.ndarray) and embeddings[idx].size > 0:
                text_value = text[idx] if idx < len(text) else np.nan

                # Log if text is missing
                # if pd.isna(text_value) or not isinstance(text_value, str) or not text_value.strip():
                #     print(f Empty or invalid text at [{category}][{idx}] → Topic: {topics[idx]}")

                test_cases.append({
                    "Category": category,
                    "Index": idx,
                    "Global_Index": f"{category}__{idx}",
                    "Embedding": embeddings[idx],
                    "Topic": topics[idx],
                    "Age": ages[idx] if idx < len(ages) else np.nan,
                    "Sex": sexes[idx] if idx < len(sexes) else np.nan,
                  
                })
                selected_topics_set.add(topics[idx])
                query_indices.append((category, idx))  # Use (cat, idx) for uniqueness

    test_df = pd.DataFrame(test_cases)
    test_df.dropna(subset=['Embedding'], inplace=True)

    # print(f" Assigned {len(test_df)} test cases from '{frequency_group}' group "
    #       f"({len(selected_topics_set)} unique topics) across {len(embedding_data)} categories (Seed: {random_seed}).")

    return test_df, query_indices


In [ ]:
# def assign_test_cases_by_frequency(embedding_data, frequency_group='top', num_topics=100, num_test_cases=30, random_seed=40):
#     """
#     Selects a subset of test cases from specified frequency groups (top, middle, bottom),
#     ensuring each topic appears only once in the final test set.
#     This version ensures FAISS retrieval integrity and balanced category selection.

#     Parameters:
#     - embedding_data: Dictionary containing embeddings and metadata per category.
#     - frequency_group: Frequency group to sample from ('top', 'middle', 'bottom').
#     - num_topics: Number of topics to select from the frequency group.
#     - num_test_cases: Number of test cases to return.
#     - random_seed: Seed for reproducibility.

#     Returns:
#     - test_df: DataFrame with unique test cases.
#     - query_indices: Indices of test cases (to exclude from retrieval corpus).
#     """

#     np.random.seed(random_seed)

#     # Step 1: Compute topic frequencies across all categories
#     topic_counts = Counter()
#     for data in embedding_data.values():
#         topic_counts.update(data.get("topic", []))

#     sorted_topics = sorted(topic_counts.items(), key=lambda x: x[1], reverse=True)
#     total_topics = len(sorted_topics)

#     # Step 2: Select topics based on frequency group
#     if frequency_group == 'top':
#         selected_topics = {topic for topic, _ in sorted_topics[:num_topics]}
#     elif frequency_group == 'bottom':
#         selected_topics = {topic for topic, _ in sorted_topics[-num_topics:]}
#     elif frequency_group == 'middle':
#         mid_start = (total_topics - num_topics) // 2
#         selected_topics = {topic for topic, _ in sorted_topics[mid_start:mid_start + num_topics]}
#     else:
#         raise ValueError("frequency_group must be 'top', 'middle', or 'bottom'")

#     # Step 3: Collect valid test cases across all categories (balanced sampling)
#     test_cases = []
#     selected_topics_set = set()  # Track selected topics to ensure uniqueness
#     query_indices = []  # To store indices for FAISS exclusion

#     for category, data in embedding_data.items():
#         embeddings = np.array(data.get("embeddings", []), dtype=object)
#         topics = np.array(data.get("topic", []), dtype=object)
#         ages = np.array(data.get("age", []), dtype=object)
#         sexes = np.array(data.get("sex", []), dtype=object)
#         text = np.array(data.get("text", []), dtype=object)

#         if "text" not in data:
#             print(f"⚠️  Missing 'text' field in category '{category}' — assigning NaN to all texts.")
#             text = np.array([np.nan] * len(embeddings), dtype=object)
#         else:
#             text = np.array(data["text"], dtype=object)

#         if len(text) != len(embeddings):
#             print(f"⚠️  Mismatch in '{category}': {len(text)} texts vs {len(embeddings)} embeddings")


#         valid_indices = [i for i, topic in enumerate(topics) if topic in selected_topics and topic not in selected_topics_set]
        
#         np.random.shuffle(valid_indices)  # Randomize selection

#         for idx in valid_indices[:num_test_cases // len(embedding_data)]:
#             if idx < len(embeddings) and isinstance(embeddings[idx], np.ndarray) and embeddings[idx].size > 0:
#                 test_cases.append({
#                     "Category": category,
#                     "Index": idx,
#                     "Embedding": embeddings[idx],
#                     "Topic": topics[idx],
#                     "Age": ages[idx] if idx < len(ages) else np.nan,
#                     "Sex": sexes[idx] if idx < len(sexes) else np.nan,
#                     "Text": text[idx] if idx < len(text) else np.nan
#                 })
#                 selected_topics_set.add(topics[idx])
#                 query_indices.append(idx)

#     # Convert to DataFrame and drop NaNs
#     test_df = pd.DataFrame(test_cases)
#     test_df.dropna(subset=['Embedding'], inplace=True)

#     print(f"Assigned {len(test_df)} test cases from {frequency_group} {num_topics} topics across {len(embedding_data)} categories (Seed: {random_seed}).")

#     return test_df, query_indices


In [ ]:
def build_faiss_index(embeddings, query_indices=None, index_type="FlatIP"):
    """
    Builds a FAISS index with optional exclusion of query cases.
 
    Returns:
    - FAISS index.
    """
    # Ensure embeddings are in the correct format
    embeddings = np.array(embeddings, dtype=np.float32)

    # Normalize embeddings BEFORE removing test queries
    faiss.normalize_L2(embeddings)  

    # Exclude test queries (if any)
    if query_indices is not None:
        if len(query_indices) >= len(embeddings):  # Avoid deleting all embeddings
            raise ValueError("Query exclusion removed all available embeddings!")
        embeddings = np.delete(embeddings, query_indices, axis=0)

    if len(embeddings) == 0:
        raise ValueError("No embeddings left to index after exclusion!")

    
    index = faiss.IndexFlatIP(embeddings.shape[1])  # Inner product for cosine similarity
        
    # Add normalized embeddings to FAISS index
    index.add(embeddings)

    return index


In [ ]:
def find_nearest_neighbors(index, query_embeddings, top_k=10):
    """
    Finds the top-K nearest neighbors for given query embeddings using FAISS.
    """
    query_embeddings = np.array(query_embeddings, dtype=np.float32)
    faiss.normalize_L2(query_embeddings)  # Normalize for cosine similarity

    distances, indices = index.search(query_embeddings, top_k + 1)  # Retrieve neighbors
    return distances[:, 1:], indices[:, 1:]  # ✅ Skip first match (query itself)

In [ ]:
def get_top_topics_per_case(test_df, category, faiss_index, topics_array, top_k=10):
    """
    Retrieves the top-K topics for each test case in the given category using FAISS.

    Parameters:
    - test_df: DataFrame containing test queries (must include 'Embedding', 'Index', 'Topic', 'Category').
    - category: The current category being evaluated.
    - faiss_index: FAISS index built from that category’s corpus (excluding test queries).
    - topics_array: List/array of all topics in the corpus for the given category.
    - top_k: Number of neighbors to retrieve.

    Returns:
    - List of dictionaries with query index, topic, and top-k retrieved topic-similarity pairs.
    """
    # Step 1: Filter test cases for current category
    category_test_cases = test_df[test_df["Category"] == category]

    # Step 2: Collect valid embeddings
    test_embeddings = category_test_cases["Embedding"].dropna().tolist()
    test_embeddings = [
        emb for emb in test_embeddings 
        if isinstance(emb, np.ndarray) and emb.size > 0
    ]

    if not test_embeddings:
        print(f"Warning: No valid embeddings for test cases in category '{category}'. Skipping.")
        return []

    # Step 3: Stack and normalize query embeddings
    query_embeddings = np.vstack(test_embeddings)

    # Step 4: Run FAISS nearest neighbor search (excludes query itself if top_k+1 used upstream)
    distances, indices = find_nearest_neighbors(faiss_index, query_embeddings, top_k)

    # Step 5: Collect retrieval results
    case_topic_results = []

    # Zip ensures that test case info is aligned with embeddings
    for i, (test_index, query_topic) in enumerate(zip(category_test_cases["Index"], category_test_cases["Topic"])):
        neighbor_indices = indices[i]

        # Filter out invalid neighbors and self-matches by index
        valid_neighbors = [
            (topics_array[idx], distances[i][j])
            for j, idx in enumerate(neighbor_indices)
            if idx < len(topics_array)  # index valid
        ][:top_k]

        case_topic_results.append({
            "Query_Index": test_index,
            "Query_Topics": query_topic,
            "Retrieved_Topics": valid_neighbors
        })

    return case_topic_results


In [ ]:
def rank_diseases_by_avg_similarity(test_df, categories, embedding_data, top_k=10):
    """
    Computes the average FAISS similarity for retrieved diseases per query across categories.

    Returns a ranked DataFrame of retrieved topics with similarity scores,
    grouped by query and retrieval category.
    """
    test_df.dropna(subset=['Embedding', 'Topic'], inplace=True)
    results = []

    for cat in categories:
        if cat not in embedding_data:
            print(f"Skipping missing category '{cat}'")
            continue

        # Select test cases for current category
        valid_cases = test_df[
            (test_df["Category"] == cat) &
            (test_df["Embedding"].apply(lambda x: isinstance(x, np.ndarray) and np.any(x != 0)))
        ]
        if valid_cases.empty:
            continue

        # Build index excluding test queries from retrieval pool
        query_indices = valid_cases["Index"].tolist()
        faiss_index = build_faiss_index(embedding_data[cat]["embeddings"], query_indices=query_indices)

        topics_array = np.array(embedding_data[cat]["topic"])
        category_results = get_top_topics_per_case(valid_cases, cat, faiss_index, topics_array, top_k=top_k)

        if not category_results:
            continue

        for result in category_results:
            query_index = result["Query_Index"]
            query_topic = result["Query_Topics"]
            retrieved_topics = result["Retrieved_Topics"]

            for disease, similarity in retrieved_topics:
                results.append({
                    "Query_Index": query_index,
                    "Query_Topic": query_topic,
                    "Retrieved_Topic": disease.lower(),
                    "Similarity": similarity,
                    "Retrieval_Category": cat  # ✅ preserve which category retrieved this topic
                })

    if not results:
        print("No retrievals found.")
        return pd.DataFrame()

    results_df = pd.DataFrame(results)

    # Get average similarity per (Query, Retrieved_Topic, Category)
    grouped_df = results_df.groupby(
        ["Query_Index", "Retrieved_Topic", "Retrieval_Category"]
    ).agg({
        "Similarity": "mean",
        "Query_Topic": "first"
    }).reset_index()

    # Now aggregate *across categories* for final top-k (averaging over all retrievals)
    final_avg_df = grouped_df.groupby(["Query_Index", "Retrieved_Topic"]).agg({
        "Similarity": "mean",
        "Query_Topic": "first"
    }).reset_index()

    # Rank top-k per query
    ranked_df = final_avg_df.groupby("Query_Index", group_keys=False).apply(
        lambda x: x.sort_values("Similarity", ascending=False).head(top_k)
    ).reset_index(drop=True)

    return ranked_df[["Query_Index", "Query_Topic", "Retrieved_Topic", "Similarity"]]


In [ ]:
high_frequency_test_df, _ = assign_test_cases_by_frequency(
    embeddings_data, frequency_group='top', num_topics=100, num_test_cases=100, random_seed=42
)

low_frequency_test_df, _  = assign_test_cases_by_frequency(
    embeddings_data, frequency_group='bottom', num_topics=100, num_test_cases=100, random_seed=42
)


medium_frequency_test_df, _  = assign_test_cases_by_frequency(
    embeddings_data, frequency_group='middle', num_topics=100, num_test_cases=100, random_seed=42
) 

In [ ]:
high_ranked_disease_df = rank_diseases_by_avg_similarity(high_frequency_test_df, categories, embedding_data, top_k=100)
medium_ranked_disease_df = rank_diseases_by_avg_similarity(medium_frequency_test_df, categories, embedding_data, top_k=100)
low_ranked_disease_df = rank_diseases_by_avg_similarity(low_frequency_test_df, categories, embedding_data, top_k=100)

In [ ]:
def partial_match(query, retrieved):
    query_set = set(map(str.strip, query.lower().split(',')))
    retrieved_set = set(map(str.strip, retrieved.lower().split(',')))
    return len(query_set & retrieved_set) > 0

def evaluate_ir_metrics(df, top_k=10):
    from sklearn.metrics import ndcg_score

    df = df.copy()
    df["Query_Topic"] = df["Query_Topic"].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
    
    # ✅ Ensure correct ranking
    df = df.sort_values(["Query_Topic", "Similarity"], ascending=[True, False])

    metrics = defaultdict(lambda: {"MRR": 0, f"Precision@{top_k}": 0, f"NDCG@{top_k}": 0, "Query_Occurrence": 0})

    for topic, group in df.groupby("Query_Topic"):
        retrieved = group["Retrieved_Topic"].tolist()[:top_k]

        relevance_vector = np.array([
            1 if partial_match(topic, retrieved_topic) else 0
            for retrieved_topic in retrieved
        ])

        if relevance_vector.any():
            mrr = 1 / (1 + np.argmax(relevance_vector))
            precision_at_k = relevance_vector.sum() / top_k
            ndcg = ndcg_score([relevance_vector], [relevance_vector], k=top_k)
        else:
            mrr = 0
            precision_at_k = 0
            ndcg = 0

        metrics[topic]["MRR"] += mrr
        metrics[topic][f"Precision@{top_k}"] += precision_at_k
        metrics[topic][f"NDCG@{top_k}"] += ndcg
        metrics[topic]["Query_Occurrence"] += 1

    for topic in metrics:
        occ = metrics[topic]["Query_Occurrence"]
        for metric in ["MRR", f"Precision@{top_k}", f"NDCG@{top_k}"]:
            metrics[topic][metric] /= occ

    return pd.DataFrame.from_dict(metrics, orient="index").reset_index().rename(columns={"index": "Query_Topics"})


In [ ]:
high_df = evaluate_ir_metrics(high_ranked_disease_df, top_k=50)

In [ ]:
medium_df = evaluate_ir_metrics(medium_ranked_disease_df , top_k=50)

In [ ]:
low_df = evaluate_ir_metrics(low_ranked_disease_df , top_k=50)

In [ ]:
high_means = high_df[['MRR', 'Precision@50', 'NDCG@50']].mean()

In [ ]:
medium_means = medium_df[['MRR', 'Precision@50', 'NDCG@50']].mean()


In [ ]:
low_means = low_df[['MRR', 'Precision@50', 'NDCG@50']].mean()

In [ ]:
import altair as alt
mean_df = pd.DataFrame({
    'Metric': ['MRR', 'Precision@50', 'NDCG@50'] * 3,
    'Score': pd.concat([high_means, medium_means, low_means]).values,
    'Frequency_Group': ['High'] * 3 + ['Medium'] * 3 + ['Low'] * 3
})

chart = alt.Chart(mean_df).mark_bar().encode(
    x=alt.X('Metric:N', title=None, axis=alt.Axis(labelAngle=45, titleFontSize=28, labelFontSize=28)),
    y=alt.Y('Score:Q', title='Mean Score',axis=alt.Axis(labelAngle=45, titleFontSize=28, labelFontSize=28)),
    color=alt.Color('Frequency_Group:N',  legend=alt.Legend(title='Frequency Group', titleFontSize=15, labelFontSize=20)),
    xOffset='Frequency_Group:N'
).properties(

    width=500,
    height=400
)

chart.save("graphs/mean_IR.pdf")
chart.show()

In [ ]:
high_df

In [ ]:
high_df[""]